In [97]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)



In [98]:
train_df = pd.read_csv("train_base_preprocessed.csv")
test_df  = pd.read_csv("test_base_preprocessed.csv")

In [99]:
train_df.rename(columns={"awards_won?": "awards_won"}, inplace=True)
test_df.rename(columns={"awards_won?": "awards_won"}, inplace=True)

In [100]:
train_df.head()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5,8,0,49,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5,4,0,60,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3,7,0,50,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1,10,0,50,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3,2,0,73,0


In [101]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   employee_id           54808 non-null  int64 
 1   department            54808 non-null  object
 2   region                54808 non-null  object
 3   education             54808 non-null  object
 4   gender                54808 non-null  object
 5   recruitment_channel   54808 non-null  object
 6   no_of_trainings       54808 non-null  int64 
 7   age                   54808 non-null  int64 
 8   previous_year_rating  54808 non-null  int64 
 9   length_of_service     54808 non-null  int64 
 10  awards_won            54808 non-null  int64 
 11  avg_training_score    54808 non-null  int64 
 12  is_promoted           54808 non-null  int64 
dtypes: int64(8), object(5)
memory usage: 5.4+ MB


In [102]:
target_col = "is_promoted"

X = train_df.drop(columns=[target_col , "employee_id"])
y = train_df[target_col]

In [103]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(exclude="object").columns.tolist()

In [104]:
print( categorical_cols )
print(numerical_cols)

['department', 'region', 'education', 'gender', 'recruitment_channel']
['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'awards_won', 'avg_training_score']


In [105]:

# stratify --> Maintains same class distribution in train and validation sets as in the original dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [106]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

In [107]:
evaluation = dict()

def evaluate_model(model, X_test, y_test, algo_name):
    y_pred = model.predict(X_test)

    

    evaluation[algo_name] = {
        "classification_report": classification_report(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }

    for key, value in evaluation.items():
        print(key)
        print(f"accuracy: {value['accuracy']}")
        print(f"precision: {value['precision']}")
        print(f"recall: {value['recall']}")
        print(f"f1-score: {value['f1_score']}")
        print("===========================================")

In [108]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [109]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

lr_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['department', 'region',
                                                   'education', 'gender',
                                                   'recruitment_channel']),
                                                 ('num', StandardScaler(),
                                                  ['no_of_trainings', 'age',
                                                   'previous_year_rating',
                                                   'length_of_service',
                                                   'awards_won',
                                                   'avg_training_score'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1))])

In [110]:
evaluate_model(lr_pipeline, X_test, y_test, "Logistic Regression")

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103


In [111]:
from sklearn.ensemble import RandomForestClassifier

In [117]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['department', 'region',
                                                   'education', 'gender',
                                                   'recruitment_channel']),
                                                 ('num', StandardScaler(),
                                                  ['no_of_trainings', 'age',
                                                   'previous_year_rating',
                                                   'length_of_service',
                                                   'awards_won',
                                                   'avg_training_score'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [118]:
evaluate_model(rf_pipeline, X_test, y_test, "Random Forest")

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855


In [119]:
from sklearn.ensemble import GradientBoostingClassifier

In [120]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ]
)

gb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['department', 'region',
                                                   'education', 'gender',
                                                   'recruitment_channel']),
                                                 ('num', StandardScaler(),
                                                  ['no_of_trainings', 'age',
                                                   'previous_year_rating',
                                                   'length_of_service',
                                                   'awards_won',
                                                   'avg_training_score'])])),
                ('model', GradientBoostingClassifier(random_state=42))])

In [121]:
evaluate_model(gb_pipeline, X_test, y_test, "Gradient Boosting")

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434


In [122]:
from sklearn.model_selection import GridSearchCV

In [123]:
rf_param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [124]:
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [125]:
rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['department',
                                                                          'region',
                                                                          'education',
                                                                          'gender',
                                                                          'recruitment_channel']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['no_of_trainings',
                                                                          'age',
                                                                          'previous_year_rating',
                                                                          'length_of_service',
                                                                          'awards_won',
                                                                          'avg_training_score'])])),
                                       ('model',
                                        RandomForestClassifier(class_weight='balanced',
                                                               n_estimators=200,
                                                               n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [None, 10, 20],
                         'model__min_samples_leaf': [1, 2],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [200, 300]},
             scoring='f1', verbose=1)

In [126]:
best_rf = rf_grid.best_estimator_

evaluate_model(best_rf, X_test, y_test, "Random Forest (Tuned)")

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434
Random Forest (Tuned)
accuracy: 0.9212734902390075
precision: 0.5522827687776142
recall: 0.4014989293361884
f1-score: 0.4649721016738996


In [127]:
gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    "model__subsample": [0.8, 1.0]
}

In [128]:
gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [129]:
gb_grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['department',
                                                                          'region',
                                                                          'education',
                                                                          'gender',
                                                                          'recruitment_channel']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['no_of_trainings',
                                                                          'age',
                                                                          'previous_year_rating',
                                                                          'length_of_service',
                                                                          'awards_won',
                                                                          'avg_training_score'])])),
                                       ('model',
                                        GradientBoostingClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.05, 0.1],
                         'model__max_depth': [3, 5],
                         'model__n_estimators': [100, 200],
                         'model__subsample': [0.8, 1.0]},
             scoring='f1', verbose=2)

In [130]:
print("Best GB Params:")
print(gb_grid.best_params_)

Best GB Params:
{'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.8}


In [131]:
best_gb = gb_grid.best_estimator_

evaluate_model(best_gb, X_test, y_test, "Gradient Boosting (Tuned)")

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434
Random Forest (Tuned)
accuracy: 0.9212734902390075
precision: 0.5522827687776142
recall: 0.4014989293361884
f1-score: 0.4649721016738996
Gradient Boosting (Tuned)
accuracy: 0.941525269111476
precision: 0.8991825613079019
recall: 0.3533190578158458
f1-score: 0.5073020753266718


In [132]:
from sklearn.ensemble import VotingClassifier


In [133]:
voting_clf = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    voting="soft",
    n_jobs=-1
)

In [134]:
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('rf',
                              Pipeline(steps=[('preprocess',
                                               ColumnTransformer(transformers=[('cat',
                                                                                OneHotEncoder(handle_unknown='ignore'),
                                                                                ['department',
                                                                                 'region',
                                                                                 'education',
                                                                                 'gender',
                                                                                 'recruitment_channel']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['no_of_trainings',
                                                                                 'age',
                                                                                 'previous_year_rating',
                                                                                 'length_of_service',
                                                                                 'awards_won',
                                                                                 'avg_training_score'])])),
                                              ('model'...
                                                                                OneHotEncoder(handle_unknown='ignore'),
                                                                                ['department',
                                                                                 'region',
                                                                                 'education',
                                                                                 'gender',
                                                                                 'recruitment_channel']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['no_of_trainings',
                                                                                 'age',
                                                                                 'previous_year_rating',
                                                                                 'length_of_service',
                                                                                 'awards_won',
                                                                                 'avg_training_score'])])),
                                              ('model',
                                               GradientBoostingClassifier(max_depth=5,
                                                                          n_estimators=200,
                                                                          random_state=42,
                                                                          subsample=0.8))]))],
                 n_jobs=-1, voting='soft')

In [135]:
evaluate_model(
    voting_clf,
    X_test,
    y_test,
    "Voting Classifier (RF + GB)"
)

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434
Random Forest (Tuned)
accuracy: 0.9212734902390075
precision: 0.5522827687776142
recall: 0.4014989293361884
f1-score: 0.4649721016738996
Gradient Boosting (Tuned)
accuracy: 0.941525269111476
precision: 0.8991825613079019
recall: 0.3533190578158458
f1-score: 0.5073020753266718
Voting Classifier (RF + GB)
accuracy: 0.9397007845283707
precision: 0.8321167883211679
recall: 0.36616702355460384
f1-score: 0.5085501858736059


#### *Oberservation:* “After tuning individual models, I used soft voting to combine Random Forest and Gradient Boosting,leveraging complementary strengths in recall and precision.”

In [136]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression


In [137]:
meta_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

In [138]:
stacking_clf = StackingClassifier(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    final_estimator=meta_model,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

In [139]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(estimators=[('rf',
                                Pipeline(steps=[('preprocess',
                                                 ColumnTransformer(transformers=[('cat',
                                                                                  OneHotEncoder(handle_unknown='ignore'),
                                                                                  ['department',
                                                                                   'region',
                                                                                   'education',
                                                                                   'gender',
                                                                                   'recruitment_channel']),
                                                                                 ('num',
                                                                                  StandardScaler(),
                                                                                  ['no_of_trainings',
                                                                                   'age',
                                                                                   'previous_year_rating',
                                                                                   'length_of_service',
                                                                                   'awards_won',
                                                                                   'avg_training_score'])])),
                                                ('mode...
                                                                                   'recruitment_channel']),
                                                                                 ('num',
                                                                                  StandardScaler(),
                                                                                  ['no_of_trainings',
                                                                                   'age',
                                                                                   'previous_year_rating',
                                                                                   'length_of_service',
                                                                                   'awards_won',
                                                                                   'avg_training_score'])])),
                                                ('model',
                                                 GradientBoostingClassifier(max_depth=5,
                                                                            n_estimators=200,
                                                                            random_state=42,
                                                                            subsample=0.8))]))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      max_iter=1000),
                   n_jobs=-1, stack_method='predict_proba')

In [140]:
evaluate_model(
    stacking_clf,
    X_test,
    y_test,
    "Stacking Classifier (RF + GB → LR)"
)

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434
Random Forest (Tuned)
accuracy: 0.9212734902390075
precision: 0.5522827687776142
recall: 0.4014989293361884
f1-score: 0.4649721016738996
Gradient Boosting (Tuned)
accuracy: 0.941525269111476
precision: 0.8991825613079019
recall: 0.3533190578158458
f1-score: 0.5073020753266718
Voting Classifier (RF + GB)
accuracy: 0.9397007845283707
precision: 0.8321167883211679
recall: 0.36616702355460384
f1-score: 0.5085501858736059
Stacking Classifier (RF + GB → LR)
accuracy: 0.8524904214559387
precision: 0.310803324099723
recall: 0.6006423982869379
f1-score: 0.40963855421686746


In [141]:
from sklearn.ensemble import AdaBoostClassifier

In [142]:
ada_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42
        ))
    ]
)

In [143]:
ada_pipeline.fit(X_train, y_train)

C:\Users\santh\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['department', 'region',
                                                   'education', 'gender',
                                                   'recruitment_channel']),
                                                 ('num', StandardScaler(),
                                                  ['no_of_trainings', 'age',
                                                   'previous_year_rating',
                                                   'length_of_service',
                                                   'awards_won',
                                                   'avg_training_score'])])),
                ('model',
                 AdaBoostClassifier(learning_rate=0.05, n_estimators=200,
                                    random_state=42))])

In [144]:
evaluate_model(
    ada_pipeline,
    X_test,
    y_test,
    "AdaBoost"
)

Logistic Regression
accuracy: 0.7630906768837803
precision: 0.21475128644939967
recall: 0.6702355460385439
f1-score: 0.32527929332294103
Random Forest
accuracy: 0.9343185550082101
precision: 0.8282208588957055
recall: 0.2890792291220557
f1-score: 0.42857142857142855
Gradient Boosting
accuracy: 0.9392446633825944
precision: 0.9652777777777778
recall: 0.29764453961456105
f1-score: 0.45499181669394434
Random Forest (Tuned)
accuracy: 0.9212734902390075
precision: 0.5522827687776142
recall: 0.4014989293361884
f1-score: 0.4649721016738996
Gradient Boosting (Tuned)
accuracy: 0.941525269111476
precision: 0.8991825613079019
recall: 0.3533190578158458
f1-score: 0.5073020753266718
Voting Classifier (RF + GB)
accuracy: 0.9397007845283707
precision: 0.8321167883211679
recall: 0.36616702355460384
f1-score: 0.5085501858736059
Stacking Classifier (RF + GB → LR)
accuracy: 0.8524904214559387
precision: 0.310803324099723
recall: 0.6006423982869379
f1-score: 0.40963855421686746
AdaBoost
accuracy: 0.924101

#### *Oberservation:* “I evaluated individual models, tuned them, and then used ensemble techniques like soft voting and stacking. The final model was selected based on F1 score due to class imbalance. ”

#### Retrain VotingClassifier on FULL train dataset

In [145]:
X_full = train_df.drop(columns=[target_col , "employee_id"])
y_full = train_df[target_col]

In [146]:
final_voting_model = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    voting="soft",
    n_jobs=-1
)

final_voting_model.fit(X_full, y_full)

VotingClassifier(estimators=[('rf',
                              Pipeline(steps=[('preprocess',
                                               ColumnTransformer(transformers=[('cat',
                                                                                OneHotEncoder(handle_unknown='ignore'),
                                                                                ['department',
                                                                                 'region',
                                                                                 'education',
                                                                                 'gender',
                                                                                 'recruitment_channel']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['no_of_trainings',
                                                                                 'age',
                                                                                 'previous_year_rating',
                                                                                 'length_of_service',
                                                                                 'awards_won',
                                                                                 'avg_training_score'])])),
                                              ('model'...
                                                                                OneHotEncoder(handle_unknown='ignore'),
                                                                                ['department',
                                                                                 'region',
                                                                                 'education',
                                                                                 'gender',
                                                                                 'recruitment_channel']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['no_of_trainings',
                                                                                 'age',
                                                                                 'previous_year_rating',
                                                                                 'length_of_service',
                                                                                 'awards_won',
                                                                                 'avg_training_score'])])),
                                              ('model',
                                               GradientBoostingClassifier(max_depth=5,
                                                                          n_estimators=200,
                                                                          random_state=42,
                                                                          subsample=0.8))]))],
                 n_jobs=-1, voting='soft')

In [147]:
X_test_final = test_df.drop(columns=["employee_id"])

In [148]:
test_predictions = final_voting_model.predict(X_test_final)

In [149]:
submission = pd.DataFrame({
    "employee_id": test_df["employee_id"],
    "is_promoted": test_predictions
})

submission.to_csv("submission.csv", index=False)

#### *Oberservation:* “By evaluating multiple baseline and tuned models, handled class imbalance using class-weighted learning, and finalized a soft voting ensemble of Random Forest and Gradient Boosting based on F1 score.”

In [153]:
train_df.rename(columns={"awards_won?": "awards_won"}, inplace=True)
test_df.rename(columns={"awards_won?": "awards_won"}, inplace=True)


In [158]:
pipeline.named_steps["preprocessing"].feature_names_in_

array(['department', 'region', 'education', 'gender',
       'recruitment_channel', 'no_of_trainings', 'age',
       'previous_year_rating', 'length_of_service', 'awards_won',
       'avg_training_score'], dtype=object)

In [157]:
X_train.columns

Index(['department', 'region', 'education', 'gender', 'recruitment_channel',
       'no_of_trainings', 'age', 'previous_year_rating', 'length_of_service',
       'awards_won', 'avg_training_score'],
      dtype='object')

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    voting="soft",
    n_jobs=-1
)

In [161]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

num_features = [
    "no_of_trainings", "age", "previous_year_rating",
    "length_of_service", "avg_training_score", "awards_won"
]

cat_features = [
    "department", "region", "education",
    "gender", "recruitment_channel"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

voting_clf = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ],
    voting="soft"
)

pipeline = Pipeline([
    ("model", voting_clf)
])

pipeline.fit(X_train, y_train)

import joblib
joblib.dump(pipeline, "promotion_voting_pipeline.pkl")


['promotion_voting_pipeline.pkl']

In [155]:
train_df.columns

Index(['employee_id', 'department', 'region', 'education', 'gender',
       'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating',
       'length_of_service', 'awards_won', 'avg_training_score', 'is_promoted'],
      dtype='object')